# Replicating the ZF-HuBERT results from scratch

You are handed a folder containing a model and a claim. This notebook rebuilds the claim from
the model, so you can decide whether to believe it.

The published numbers are:

| result | value | baseline |
|---|---|---|
| call type, 8-way, linear probe, leave-birds-out | **0.811** accuracy | 0.207 majority |
| call type, unsupervised k-means, AMI | **0.547** at k=16 | 0.002 (null 95th pct) |
| same, recomputed within each bird | **0.707** | — |

**There are two ways to run this notebook and they prove different things.**

- **Mode A — from audio.** You have the 2814 curated clips (26 birds). The notebook runs them through the
  model itself and recomputes everything. This replicates the *whole* pipeline.
- **Mode B — from cached embeddings.** You only have `data/run11_layersweep.npz`. The notebook
  replicates the *analysis* but takes the feature extraction on trust.

Mode B is not worthless — the analysis is where most mistakes live — but it cannot catch a bug
in how audio became vectors. The notebook detects which mode it is in and says so. Either way,
step 2 checks the model you loaded against the shipped embeddings, so Mode B still verifies
that your copy of the weights is the one the numbers came from.


## 0. Getting the folder

On Savio, no download needed:

```bash
cp -r /global/home/users/jonathanswang/zf_hubert_run11 ~/
```

From your own machine:

```bash
rsync -avP $USER@hpc.brc.berkeley.edu:/global/home/users/jonathanswang/zf_hubert_run11 .
cd zf_hubert_run11 && shasum -a 256 -c SHA256SUMS   # confirm nothing corrupted in transit
```

Then put this notebook anywhere inside that folder (or next to it) and run it.

Requirements: `torch`, `torchaudio`, `numpy`, `scikit-learn`. Mode A also needs the clip
directory. No GPU required — everything below runs on a laptop CPU.


In [1]:
import os, sys, csv, time, warnings
from pathlib import Path
import numpy as np
warnings.filterwarnings("ignore")

# --- find the bundle, wherever this notebook happens to be sitting -------------------
CANDIDATES = [
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
    Path.home() / "zf_hubert_run11",
    Path("/global/home/users/jonathanswang/zf_hubert_run11"),                # Savio (shared)
    Path("/global/scratch/users/jonathanswang/release/zf_hubert_run11"),     # Savio (owner only)
]
def find(rel):
    for base in CANDIDATES:
        try:
            p = (base / rel)
            if p.exists():
                return p.resolve()
        except (PermissionError, OSError):
            continue
    return None

WEIGHTS = find("weights/zf_hubert_run11_encoder.pt")
CACHED  = find("data/run11_layersweep.npz")
LABELS  = find("data/calltype_labels.csv")
EXAMPLES= find("examples")

for nm, p in [("weights", WEIGHTS), ("cached embeddings", CACHED),
              ("labels csv", LABELS), ("example wavs", EXAMPLES)]:
    print(f"{nm:20s} {p if p else '** NOT FOUND **'}")

if WEIGHTS is None:
    raise SystemExit("Could not find the weights. Add the bundle path to CANDIDATES above.")


weights              /Users/jonathanwang/Desktop/vocalizations_lab/release/zf_hubert_run11/weights/zf_hubert_run11_encoder.pt
cached embeddings    /Users/jonathanwang/Desktop/vocalizations_lab/release/zf_hubert_run11/data/run11_layersweep.npz
labels csv           /Users/jonathanwang/Desktop/vocalizations_lab/.claude/worktrees/bird-count-correction/release/zf_hubert_run11/data/calltype_labels.csv
example wavs         /Users/jonathanwang/Desktop/vocalizations_lab/release/zf_hubert_run11/examples


### Where is the audio?

Mode A needs the 2814 curated clips (26 birds). They are *not* in the bundle — only 16 examples are.
Point `CLIP_DIR` at them if you have them; the cell searches a few likely places first and
falls back to Mode B if it finds nothing.


In [2]:
# Add your own path here if you know where the curated clips live.
CLIP_DIR_CANDIDATES = [
    os.environ.get("ZF_CLIP_DIR"),
    "/global/scratch/users/jonathanswang/data/VocExtracts",
    "/global/home/users/jonathanswang/VocExtracts",
    Path.home() / "VocExtracts",
]

wanted = [r["fname"] for r in csv.DictReader(open(LABELS))] if LABELS else []
print(f"{len(wanted)} clip names listed in the labels file")

CLIP_DIR, MODE = None, "B"
for cand in CLIP_DIR_CANDIDATES:
    if not cand:
        continue
    p = Path(cand)
    try:
        if p.is_dir():
            present = sum((p / w).exists() for w in wanted[:200])   # sample before committing
            if present > 190:
                CLIP_DIR, MODE = p, "A"
                break
            print(f"  {p} exists but only {present}/200 sampled clips found - skipping")
    except (PermissionError, OSError):
        continue

print(f"\nMODE {MODE}: " + ("recomputing embeddings from audio at " + str(CLIP_DIR)
                            if MODE == "A" else
                            "no clip directory found -> using cached embeddings"))
if MODE == "B" and CACHED is None:
    raise SystemExit("Neither audio nor cached embeddings available; cannot proceed.")


2867 clip names listed in the labels file

MODE B: no clip directory found -> using cached embeddings


## 1. Load the model

The `.pt` file carries its own constructor kwargs, so there is no config file to keep in sync:
whatever architecture the weights were trained with is what gets built. `strict=True` means a
silent architecture mismatch is impossible — it would raise instead of quietly producing
garbage embeddings.

Everything here is inlined rather than imported from `zf_hubert.py`, so this notebook stands on
its own and you can see exactly what is being done to your audio.


In [3]:
import torch, torchaudio

def load_encoder(path, device="cpu"):
    obj = torch.load(path, map_location="cpu", weights_only=False)
    model = torchaudio.models.wav2vec2_model(aux_num_out=None, **obj["encoder_config"])
    model.load_state_dict(obj["state_dict"], strict=True)   # raises on any mismatch
    model.eval().to(device)
    model.zf_meta = obj.get("meta", {})
    return model

SR = 16000
_rs = {}
def load_audio(path, sample_rate=SR):
    wav, sr = torchaudio.load(str(path))
    if sr != sample_rate:
        _rs.setdefault((sr, sample_rate), torchaudio.transforms.Resample(sr, sample_rate))
        wav = _rs[(sr, sample_rate)](wav)
    return wav.mean(0, keepdim=True)          # mono = channel mean

@torch.no_grad()
def embed_frames(model, wav):
    feats, _ = model.extract_features(wav, None)      # list of 12 x (1, T, 768)
    return torch.stack([f.squeeze(0) for f in feats]).numpy()

@torch.no_grad()
def embed_file(model, path):
    return embed_frames(model, load_audio(path)).mean(axis=1)   # (12, 768), mean over time

t0 = time.time()
model = load_encoder(WEIGHTS)
n_par = sum(p.numel() for p in model.parameters())
print(f"loaded in {time.time()-t0:.1f}s | {n_par/1e6:.1f} M parameters")
print(f"torch {torch.__version__} | torchaudio {torchaudio.__version__}")


loaded in 0.4s | 94.4 M parameters
torch 2.2.2 | torchaudio 2.2.2


## 2. Does *your* model reproduce the shipped embeddings?

Before replicating any result, check that the weights you downloaded, loaded on your hardware,
produce the same vectors the published numbers were computed from. This runs the 16 example
clips through your model and compares against their rows in the cached file.

**Do not expect bitwise equality.** The cache was produced on an NVIDIA GPU; you are almost
certainly on CPU. Different BLAS kernels sum in different orders, so float32 results differ in
the last few digits. The right test is cosine similarity ≈ 1, not `array_equal`. A genuine bug
— wrong layer order, wrong normalization, wrong resampling — produces cosines of 0.3, not
0.9999. The failure modes are nowhere near each other.


In [4]:
d = np.load(CACHED, allow_pickle=True)
emb_cached, y, birds, names, classes = (d["emb"], d["y"], d["birds"], d["names"], d["classes"])
name_to_row = {n: i for i, n in enumerate(names)}
print(f"cache: {emb_cached.shape[0]} clips x {emb_cached.shape[1]} layers x {emb_cached.shape[2]} dims")

checked = []
for wav_path in sorted(Path(EXAMPLES).glob("*.wav")):
    if wav_path.name not in name_to_row:
        continue
    mine  = embed_file(model, wav_path)                       # (12, 768)
    ref   = emb_cached[name_to_row[wav_path.name]]            # (12, 768)
    cos   = float(np.mean(np.sum(mine*ref, 1) /
                          (np.linalg.norm(mine,axis=1)*np.linalg.norm(ref,axis=1))))
    rel   = float(np.max(np.abs(mine-ref)) / (np.abs(ref).max() + 1e-12))
    checked.append((wav_path.name, cos, rel))

print(f"\n{len(checked)} example clips checked (mean cosine over all 12 layers):")
for n, c, r in checked[:5]:
    print(f"  {n:42s} cos {c:.7f}   max rel diff {r:.2e}")
print("  ...")
worst = min(c for _, c, _ in checked)
print(f"\nworst cosine across all clips and layers: {worst:.7f}")
assert worst > 0.999, "Embeddings do NOT match the cache - stop here and investigate."
print("PASS - your model matches the one the published numbers came from.")


cache: 2867 clips x 12 layers x 768 dims


[W NNPACK.cpp:64] Could not initialize NNPACK! Reason: Unsupported hardware.



16 example clips checked (mean cosine over all 12 layers):
  BlaBla0506_110302-AggC-04.wav              cos 1.0000000   max rel diff 1.18e-04
  BlaBla0506_110302-AggC-05.wav              cos 0.9999999   max rel diff 1.63e-04
  BlaBla0506_110302-DC-01.wav                cos 1.0000000   max rel diff 1.90e-04
  BlaBla0506_110302-DC-02.wav                cos 1.0000000   max rel diff 1.17e-04
  BlaBla0506_110302-NekakleC-02.wav          cos 0.9999999   max rel diff 3.32e-04
  ...

worst cosine across all clips and layers: 0.9999998
PASS - your model matches the one the published numbers came from.


## 3. Build the feature matrix

In Mode A this runs all 2867 clips through the model — a few minutes on CPU. In Mode B it just
takes the cached array. Either way we end up with the same object: `(2867, 12, 768)`, one
mean-pooled vector per clip per layer, plus the call-type label and bird ID for each clip.

Note the bird ID is parsed from the filename prefix. That matters more than it looks — the
entire evaluation protocol depends on grouping by bird.


In [5]:
if MODE == "A":
    lab = {r["fname"]: r["label"] for r in csv.DictReader(open(LABELS))}
    classes = np.array(sorted(set(lab.values())))
    cls_ix  = {c: i for i, c in enumerate(classes)}
    rows, ys, bs, ns = [], [], [], []
    t0 = time.time()
    for i, fname in enumerate(wanted):
        p = CLIP_DIR / fname
        if not p.exists():
            continue
        rows.append(embed_file(model, p)); ys.append(cls_ix[lab[fname]])
        bs.append(fname.split("_")[0]); ns.append(fname)
        if (i+1) % 250 == 0:
            print(f"  {i+1}/{len(wanted)}  ({time.time()-t0:.0f}s)")
    emb   = np.stack(rows).astype(np.float32)
    y     = np.array(ys); birds = np.array(bs); names = np.array(ns)
    print(f"\nrecomputed {emb.shape[0]} clips from audio in {time.time()-t0:.0f}s")
else:
    emb = emb_cached
    print(f"using cached embeddings: {emb.shape[0]} clips")

# --- CORRECTION (2026-08-16): bird IDs are filename prefixes, not individuals ---------
# "HPiHPi4748" and "HpiHpi4748" are the same bird under two capitalizations, and four
# "Unknown*" prefixes are catch-alls rather than individuals. Left uncorrected, the first
# lets one bird sit in both train and test -- exactly what leave-birds-out exists to stop.
n_before = (len(np.unique(birds)), emb.shape[0])
birds = np.array([b.lower() for b in birds])
_named = ~np.char.startswith(np.array([str(b) for b in birds]), "unknown")
emb, y, birds, names = emb[_named], y[_named], birds[_named], names[_named]
print(f"bird-ID correction: {n_before[0]} prefixes / {n_before[1]} clips "
      f"-> {len(np.unique(birds))} birds / {emb.shape[0]} clips")

print(f"{emb.shape[0]} clips | {len(classes)} call types | {len(np.unique(birds))} birds")
counts = np.bincount(y)
print("class counts:", dict(zip(classes.tolist(), counts.tolist())))
majority = counts.max() / counts.sum()
print(f"majority-class baseline = {majority:.3f}")
assert len(np.unique(names)) == len(names), "duplicate clips would inflate every score below"


using cached embeddings: 2867 clips
bird-ID correction: 31 prefixes / 2867 clips -> 26 birds / 2814 clips
2814 clips | 8 call types | 26 birds
class counts: {'Ag': 196, 'DC': 583, 'Ne': 575, 'So': 192, 'Te': 577, 'Th': 279, 'Tu': 240, 'Wh': 172}
majority-class baseline = 0.207


## 4. Replicate the supervised result (published: 0.821)

One multinomial logistic regression per layer on the frozen embedding. The encoder gets no
gradient — the only thing fit is a 768→8 linear map. That is deliberate: it measures whether
call type is *linearly readable* from the geometry the model built without labels, not whether
the model can be trained to do the task.

**The split is the load-bearing part.** `StratifiedGroupKFold(groups=birds)` puts each bird
entirely in one fold, so every clip is predicted by a model that never heard that individual.
With a random split the probe could learn "this voice → this bird → this bird mostly makes
DCs" and score well with no call-type understanding at all. We measure that inflation in the
next cell rather than just asserting it.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

rows = []
t0 = time.time()
for L in range(emb.shape[1]):
    oof = cross_val_predict(LogisticRegression(max_iter=2000, C=1.0),
                            emb[:, L, :], y, groups=birds,
                            cv=StratifiedGroupKFold(n_splits=5), n_jobs=-1)
    rows.append((L, accuracy_score(y, oof), f1_score(y, oof, average="macro"), oof))
    print(f"layer {L:2d} | acc {rows[-1][1]:.3f} | macroF1 {rows[-1][2]:.3f}")

best = max(rows, key=lambda r: r[1])
acc  = [r[1] for r in rows]
print(f"\nbest layer {best[0]}: {best[1]:.3f}   (published 0.811 at layer 3)")
print(f"spread across all 12 layers: {max(acc)-min(acc):.3f}   ({min(acc):.3f} - {max(acc):.3f})")
print(f"elapsed {time.time()-t0:.0f}s")


layer  0 | acc 0.789 | macroF1 0.765


layer  1 | acc 0.800 | macroF1 0.776


layer  2 | acc 0.799 | macroF1 0.775


layer  3 | acc 0.811 | macroF1 0.788


layer  4 | acc 0.802 | macroF1 0.780


...

layer  7 | acc 0.796 | macroF1 0.773


layer  8 | acc 0.799 | macroF1 0.776


layer  9 | acc 0.796 | macroF1 0.774


layer 10 | acc 0.795 | macroF1 0.772


layer 11 | acc 0.791 | macroF1 0.767

best layer 3: 0.811   (published 0.811 at layer 3)
spread across all 12 layers: 0.022   (0.789 - 0.811)
elapsed 19s


### How much does the protocol actually matter?

Same features, same classifier, only the split changes. If the gap is large, every leave-birds-out
number in the README is doing real work and any random-split number you see elsewhere is not
comparable to it.


In [7]:
from sklearn.model_selection import StratifiedKFold

X3 = emb[:, 3, :]
oof_random = cross_val_predict(LogisticRegression(max_iter=2000, C=1.0), X3, y,
                               cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0),
                               n_jobs=-1)
oof_bird   = best[3] if best[0] == 3 else cross_val_predict(
                 LogisticRegression(max_iter=2000, C=1.0), X3, y, groups=birds,
                 cv=StratifiedGroupKFold(n_splits=5), n_jobs=-1)

print(f"random 5-fold split      {accuracy_score(y, oof_random):.3f}   <- optimistic")
print(f"leave-birds-out (used)   {accuracy_score(y, oof_bird):.3f}   <- what we report")
print(f"inflation from ignoring bird identity: "
      f"{accuracy_score(y, oof_random) - accuracy_score(y, oof_bird):+.3f}")

# fold-to-fold spread tells you how precise that 0.821 really is
per_fold = []
for tr, te in StratifiedGroupKFold(n_splits=5).split(X3, y, groups=birds):
    assert not (set(birds[tr]) & set(birds[te])), "bird leaked across folds"
    clf = LogisticRegression(max_iter=2000, C=1.0).fit(X3[tr], y[tr])
    per_fold.append(accuracy_score(y[te], clf.predict(X3[te])))
print(f"\nper-fold accuracy: {np.round(per_fold,3).tolist()}")
print(f"sd across folds = {np.std(per_fold):.3f}  <- larger than the whole 12-layer spread,")
print("   which is why 'layer 3 is best' is a weak preference and not a finding.")


random 5-fold split      0.925   <- optimistic
leave-birds-out (used)   0.811   <- what we report
inflation from ignoring bird identity: +0.114



per-fold accuracy: [0.828, 0.822, 0.818, 0.843, 0.745]
sd across folds = 0.034  <- larger than the whole 12-layer spread,
   which is why 'layer 3 is best' is a weak preference and not a finding.


In [8]:
cm = confusion_matrix(y, best[3], normalize="true")
print("per-class recall (layer %d):" % best[0])
for c, r in sorted(zip(classes, cm.diagonal()), key=lambda t: -t[1]):
    print(f"  {c}: {r:.3f}  (n={counts[list(classes).index(c)]})")
print("\nPublished: So 0.99, DC 0.97, Ag 0.95, Wh 0.83, Te 0.82, Ne 0.82, Th 0.68, Tu 0.39")


per-class recall (layer 3):
  So: 0.990  (n=192)
  DC: 0.964  (n=583)
  Ag: 0.949  (n=196)
  Wh: 0.837  (n=172)
  Te: 0.832  (n=577)
  Ne: 0.816  (n=575)
  Th: 0.573  (n=279)
  Tu: 0.379  (n=240)

Published: So 0.99, DC 0.97, Ag 0.95, Wh 0.83, Te 0.82, Ne 0.82, Th 0.68, Tu 0.39


## 5. Replicate the unsupervised result (published: AMI 0.547)

A harder claim than the probe. The probe gets 768 dimensions of freedom to carve boundaries and
can find a separating hyperplane even in a representation where call type is not a dominant
factor. k-means gets no such freedom — it finds whatever the dominant geometry already is. If
the clusters line up with call type, call type is a principal axis of the representation.

Two choices worth understanding rather than copying:

**L2-normalize first.** k-means uses Euclidean distance, so without normalization the handful
of dimensions with the largest variance would dictate the partition on their own.

**Score with AMI, never NMI or purity.** NMI and purity climb with k for free — on structureless
data NMI goes from 0.02 at k=4 to 0.19 at k=60 — so a sweep ranked on them just returns the
largest k you tried. AMI has expectation 0 at every k, which is what makes the sweep mean
anything. We do not force k=8: forcing k to the number of human categories measures agreement
with a taxonomy, while sweeping asks whether structure exists at all.


In [9]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import adjusted_mutual_info_score as ami

LAYER = 3
X = normalize(emb[:, LAYER, :])

def within_bird_ami(clusters, y, birds, min_clips=12, min_types=2):
    # AMI recomputed inside each bird, weighted by clip count. Birds with too few clips
    # or only one call type are dropped -- AMI is degenerate there.
    num = den = 0.0; used = 0
    for b in np.unique(birds):
        m = birds == b
        if m.sum() < min_clips or len(np.unique(y[m])) < min_types:
            continue
        num += ami(y[m], clusters[m]) * m.sum(); den += m.sum(); used += 1
    return (num/den if den else np.nan), used

res = []
for k in [2, 4, 6, 8, 10, 12, 16, 20, 24, 32, 40, 60]:
    c = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    wb, nb = within_bird_ami(c, y, birds)
    res.append((k, ami(y, c), ami(birds, c), wb, c))
    print(f"k={k:3d} | call type {res[-1][1]:.3f} | bird ID {res[-1][2]:.3f} | within-bird {wb:.3f}")

bk = max(res, key=lambda r: r[1])
print(f"\nbest k = {bk[0]}, AMI vs call type = {bk[1]:.3f}   (published 0.547 at k=16)")


k=  2 | call type 0.223 | bird ID 0.024 | within-bird 0.318


k=  4 | call type 0.359 | bird ID 0.080 | within-bird 0.523


k=  6 | call type 0.381 | bird ID 0.139 | within-bird 0.579


k=  8 | call type 0.468 | bird ID 0.179 | within-bird 0.649


k= 10 | call type 0.495 | bird ID 0.205 | within-bird 0.656


...

k= 20 | call type 0.542 | bird ID 0.272 | within-bird 0.703


k= 24 | call type 0.526 | bird ID 0.285 | within-bird 0.696


k= 32 | call type 0.509 | bird ID 0.312 | within-bird 0.681


k= 40 | call type 0.514 | bird ID 0.338 | within-bird 0.682


k= 60 | call type 0.499 | bird ID 0.381 | within-bird 0.667

best k = 16, AMI vs call type = 0.547   (published 0.547 at k=16)


### The control that decides whether any of this is real

Read the *bird ID* column before celebrating the *call type* column.

Here is the failure mode. Suppose the embedding encodes only individual identity — voice, not
call type. Suppose also that birds differ in which calls they tend to produce (they do). Then a
clustering that is purely a **bird** clustering still scores well against call type, because
each bird's cluster inherits that bird's favoured call type. Global AMI cannot tell the two
apart, and a permutation test does not rescue you: in a simulation of exactly this confound the
global AMI was 0.489 against a null 95th percentile of 0.021 — wildly "significant", and
completely wrong.

The fix is to recompute AMI **within each bird**, which holds voice constant. If the clustering
were really bird identity, every clip of a given bird would land in the same cluster, mutual
information inside that bird would be exactly 0, and the within-bird column would collapse.

It does not collapse — it goes *up*. Which raises a second question the original analysis did
not answer: is within-bird higher only because each bird uses fewer call types, making the task
easier? The matched control below settles it by resampling clips with the **same size and same
call-type mix** as each bird, but drawn from *other* birds.


In [10]:
c_best = bk[4]
wb, n_birds = within_bird_ami(c_best, y, birds)
print(f"global AMI vs call type   {bk[1]:.3f}")
print(f"global AMI vs bird ID     {bk[2]:.3f}")
print(f"within-bird AMI           {wb:.3f}   ({n_birds} birds pass the filter)")

keep = [b for b in np.unique(birds)
        if (birds == b).sum() >= 12 and len(np.unique(y[birds == b])) >= 2]
idx_by_type = {t: np.where(y == t)[0] for t in np.unique(y)}
scores = []
for seed in range(10):
    r = np.random.default_rng(seed); vals, wts = [], []
    for b in keep:
        m = birds == b; pick = []
        for t, n in zip(*np.unique(y[m], return_counts=True)):
            pool = idx_by_type[t][birds[idx_by_type[t]] != b]      # same type, other birds
            pick.append(r.choice(pool, size=min(n, len(pool)), replace=False))
        pick = np.concatenate(pick)
        vals.append(ami(y[pick], c_best[pick])); wts.append(len(pick))
    scores.append(np.average(vals, weights=wts))
print(f"matched mixed-bird control {np.mean(scores):.3f} +/- {np.std(scores):.3f}")
print("\nSame clip count and same call-type mix as each bird, but clips drawn from many birds.")
print("If within-bird >> this, holding voice constant genuinely sharpens the call-type signal")
print("and the gain is not an artifact of birds using fewer call types.")

rng = np.random.default_rng(0)
null = [ami(rng.permutation(y), c_best) for _ in range(200)]
print(f"\nlabel-shuffled null: mean {np.mean(null):+.4f}, 95th pct {np.percentile(null,95):.4f}")


global AMI vs call type   0.547
global AMI vs bird ID     0.251
within-bird AMI           0.707   (25 birds pass the filter)
matched mixed-bird control 0.508 +/- 0.010

Same clip count and same call-type mix as each bird, but clips drawn from many birds.
If within-bird >> this, holding voice constant genuinely sharpens the call-type signal
and the gain is not an artifact of birds using fewer call types.



label-shuffled null: mean -0.0001, 95th pct 0.0020


## 6. What does AMI 0.547 actually mean?

It is an information measure, not an accuracy, and it is easy to misread as "54.7% correct".
This cell converts it into quantities you can reason about.


In [11]:
from sklearn.metrics import mutual_info_score, homogeneity_score, completeness_score
from scipy.stats import entropy

b = 1/np.log(2)
Hy = entropy(np.bincount(y))*b
MI = mutual_info_score(y, c_best)*b
print(f"H(call type)           {Hy:.3f} bits  <- uncertainty before you know the cluster")
print(f"MI(call type; cluster) {MI:.3f} bits  = {MI/Hy:.1%} of it removed")
print(f"homogeneity {homogeneity_score(y,c_best):.3f} (clusters are pure)  "
      f"completeness {completeness_score(y,c_best):.3f} (types are split across clusters)")

maj = np.zeros_like(y)
for k in np.unique(c_best):
    m = c_best == k; maj[m] = np.bincount(y[m]).argmax()
print(f"\nlabel each cluster by majority vote -> {(maj==y).mean():.3f} accuracy, using zero labels")
print(f"   (vs {majority:.3f} baseline and {best[1]:.3f} for the supervised probe)")
print("   purity like this is only readable at fixed k - at k=n it is trivially 1.0")

# the ceiling: a PERFECTLY pure clustering, split to the same granularity, does not score 1.0
rng = np.random.default_rng(0); perfect = np.empty_like(y); nxt = 0
sizes = np.bincount(y); share = np.maximum(1, np.round(bk[0]*sizes/sizes.sum()).astype(int))
for t in range(len(classes)):
    ix = np.where(y == t)[0]; rng.shuffle(ix)
    for j, part in enumerate(np.array_split(ix, share[t])): perfect[part] = nxt + j
    nxt += share[t]
print(f"\nceiling: a perfectly PURE clustering split {len(np.unique(perfect))} ways scores "
      f"AMI {ami(y, perfect):.3f}, not 1.0")
print("   AMI charges you for over-splitting, so read 0.547 against ~0.80, not against 1.0.")


H(call type)           2.819 bits  <- uncertainty before you know the cluster
MI(call type; cluster) 1.842 bits  = 65.3% of it removed
homogeneity 0.653 (clusters are pure)  completeness 0.477 (types are split across clusters)

label each cluster by majority vote -> 0.744 accuracy, using zero labels
   (vs 0.207 baseline and 0.811 for the supervised probe)
   purity like this is only readable at fixed k - at k=n it is trivially 1.0

ceiling: a perfectly PURE clustering split 15 ways scores AMI 0.838, not 1.0
   AMI charges you for over-splitting, so read 0.547 against ~0.80, not against 1.0.


## 7. Verdict


In [12]:
# NOTE: these are the CORRECTED reference values (26 birds, 2814 clips). Earlier versions of
# this bundle quoted 0.821 / k=20 / 31 birds, which counted filename prefixes rather than
# individuals -- see the "Corrected" note in README.md.
PUBLISHED = {"probe acc (best layer)": 0.811, "probe best layer": 3,
             "AMI vs call type": 0.547, "AMI vs bird": 0.251,
             "within-bird AMI": 0.707, "best k": 16}
mine = {"probe acc (best layer)": best[1], "probe best layer": best[0],
        "AMI vs call type": bk[1], "AMI vs bird": bk[2],
        "within-bird AMI": wb, "best k": bk[0]}

print(f"MODE {MODE} - " + ("recomputed from audio: full replication"
                           if MODE=="A" else
                           "cached embeddings: analysis replicated, feature extraction taken on trust"))
print(f"{'quantity':26s} {'published':>10s} {'yours':>10s} {'diff':>9s}")
for k_ in PUBLISHED:
    p, m_ = PUBLISHED[k_], mine[k_]
    flag = "" if abs(p-m_) < (0.5 if isinstance(p,int) else 0.02) else "   <-- CHECK"
    print(f"{k_:26s} {p:>10} {m_:>10.3f} {m_-p:>+9.3f}{flag}")

print()
print("In Mode B these should match to ~1e-3 (same inputs, same seeds; only BLAS ordering differs).")
print("In Mode A expect small genuine differences - your embeddings are recomputed on different")
print("hardware, and k-means and the CV folds amplify tiny float changes into reassignments.")
print("A few thousandths is fine. A few hundredths means something is actually different, and")
print("the first suspects are: a clip set that is not exactly these 2867 files, resampling to")
print("something other than 16 kHz, or channel handling other than the mean.")


MODE B - cached embeddings: analysis replicated, feature extraction taken on trust
quantity                    published      yours      diff
probe acc (best layer)          0.811      0.811    -0.000
probe best layer                    3      3.000    +0.000
AMI vs call type                0.547      0.547    +0.000
AMI vs bird                     0.251      0.251    +0.000
within-bird AMI                 0.707      0.707    -0.000
best k                             16     16.000    +0.000

In Mode B these should match to ~1e-3 (same inputs, same seeds; only BLAS ordering differs).
In Mode A expect small genuine differences - your embeddings are recomputed on different
hardware, and k-means and the CV folds amplify tiny float changes into reassignments.
A few thousandths is fine. A few hundredths means something is actually different, and
the first suspects are: a clip set that is not exactly these 2867 files, resampling to
something other than 16 kHz, or channel handling other than t

## What this notebook deliberately does not do

Three baselines would strengthen the claims and none of them are here:

1. **A random-initialized encoder.** Same architecture, untrained weights, same pipeline.
   Untrained deep nets are surprisingly good feature extractors, so if trained ≈ random then
   pretraining contributed nothing. This is the single most informative missing baseline. It
   needs Mode A, since it means re-extracting every clip:
   ```python
   import copy
   obj = torch.load(WEIGHTS, map_location="cpu", weights_only=False)
   rand = torchaudio.models.wav2vec2_model(aux_num_out=None, **obj["encoder_config"]).eval()
   # then rerun section 3 with `rand` in place of `model`
   ```
2. **A duration baseline.** Call types differ in length. A clustering that merely sorts clips by
   duration can look impressive while encoding nothing acoustic.
3. **A spectrogram baseline.** Mean-pooled log-mel through the same probe and the same folds.
   If it matches 0.821, the transformer is not earning its keep.

And one caveat that no baseline fixes: **the curated clips were part of the ~100 h corpus the
model was pretrained on.** Pretraining was label-free so call-type labels cannot have been
memorized, but the model has heard this audio. Comparisons between layers, runs and baselines
are unaffected — they all see the same clips — while absolute numbers may be optimistic
relative to a fresh recording session. A pretraining holdout is the clean fix and has not been
run.
